#   mlflow model  registry

In [85]:
import  pandas  as  pd
import  numpy  as  np 
import  matplotlib.pyplot as plt 
from  sklearn.preprocessing import  StandardScaler
from  sklearn.tree import  DecisionTreeClassifier
from  sklearn.linear_model import  LogisticRegression 
from  sklearn.model_selection import  train_test_split
from  sklearn.ensemble import  RandomForestClassifier
from  sklearn.metrics import  f1_score , roc_auc_score ,  accuracy_score , confusion_matrix
import  seaborn as sns

from  sklearn.dummy import  DummyClassifier
import    mlflow



In [4]:
data =  pd.read_csv("../raw_data/heart.csv")
data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [5]:
X = data.iloc[:,:-1].values
y =  data.iloc[:,-1].values

In [8]:
X_train , X_test , y_train , y_test =  train_test_split(X ,y , test_size=0.2 ,  random_state=42 , 
                                                        stratify=y)

In [9]:
#   preprocessing 
scaler =  StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# premier benckmark  avec le model   Dummy(bete)

In [13]:
mlflow.set_experiment("benckmark")
mlflow.set_experiment_tag("Dummy" , "test1")
mlflow.sklearn.autolog()

with  mlflow.start_run(run_name="Dummyclassifier")  as run :
    model = DummyClassifier()
    model.fit(X_train , y_train)
    y_pred = model.predict(X_test)
    #   metrics
    f1 = f1_score(y_test , y_pred)
    roc_score =  roc_auc_score(y_test , y_pred)

    #  log   mlflow
    mlflow.log_metrics({"f1_score": f1,"roc_score" : roc_score })
    

2026/10/03 12:36:05 INFO mlflow.tracking.fluent: Experiment with name 'benckmark' does not exist. Creating a new experiment.
2026/10/03 12:36:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/03 12:36:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/dylan/Bureau/ml-eng/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

#   chargement  du  model  en  local 

In [23]:
model_path =  "./mlruns/9/models/m-9174b6964aca41e0820daba2c2639c98/artifacts"

model_new =  mlflow.sklearn.load_model(model_path)
print("model  bien  charge  depuis  le  path  file local !\n")
#   prediction  
print("--------------quelque prediction------------- \n")
print("valuer  predict  par  le model: " , model_new.predict(X_test[:4]))
print("valuer  vrai :" , y_test[:4])


model  bien  charge  depuis  le  path  file local !

--------------quelque prediction------------- 

valuer  predict  par  le model:  [1 1 1 1]
valuer  vrai : [0 1 0 1]


#  enregistremmt depuis  un Dossier  local 

In [26]:
mlflow.sklearn.save_model(model , "Dummy")
print("dossier  cree  avec  sucess ")
#   load  model
new_model =  mlflow.sklearn.load_model("Dummy")
print("  sucess  ok  model  bien  charge depuis  le dossier  Dummy\n")
#   prediction  
print("--------------quelque prediction------------- \n")
print("valuer  predict  par  le model: " , new_model.predict(X_test[:4]))
print("valuer  vrai :" , y_test[:4])



2026/10/03 12:51:45 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


dossier  cree  avec  sucess 
  sucess  ok  model  bien  charge depuis  le dossier  Dummy

--------------quelque prediction------------- 

valuer  predict  par  le model:  [1 1 1 1]
valuer  vrai : [0 1 0 1]


 # enregistrement  depuis  mlflow  et  chargement

In [27]:
mlflow.sklearn.autolog()

with  mlflow.start_run(run_name="Logistic")  as run :
    model = LogisticRegression()
    model.fit(X_train , y_train)
    y_pred = model.predict(X_test)
    #   metrics
    f1 = f1_score(y_test , y_pred)
    roc_score =  roc_auc_score(y_test , y_pred)

    #  log   mlflow
    mlflow.log_metrics({"f1_score": f1,"roc_score" : roc_score })
    #  log  model 
    mlflow.sklearn.log_model(model ,"traking_path")

2026/10/03 12:55:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/03 12:55:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/dylan/Bureau/ml-eng/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/10/03 12:55:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/03 12:56:14 WARNING mlflow.utils.environment: Failed to resolve installe

In [32]:
# on  recupere  le derier  run et  posible  le nom  du  run 
print(mlflow.last_active_run().info.run_id)
print(mlflow.last_active_run().info.run_name)

ec4eb4beb6ee48ab8855e585b0b02843
Logistic


#   notes    peut  etre  interessant

In [38]:
#   on peut  mettre  experiment_id  ou  name  dont  comme  benckmark ou  id = 9
df =  mlflow.search_runs(experiment_names=['benckmark'] ,
                        # experiment_ids=["9"]
                        )
df

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.training_roc_auc,metrics.roc_score,metrics.training_precision_score,metrics.training_log_loss,...,params.verbose,params.class_weight,params.constant,params.strategy,tags.estimator_name,tags.mlflow.runName,tags.mlflow.user,tags.estimator_class,tags.mlflow.source.name,tags.mlflow.source.type
0,ec4eb4beb6ee48ab8855e585b0b02843,9,FINISHED,/home/dylan/Bureau/ml-eng/mlflow_projets/mlrun...,2026-10-03 10:55:40.412000+00:00,2026-10-03 10:56:14.578000+00:00,0.922211,0.807143,0.851095,0.353031,...,0,None,NaN,NaN,LogisticRegression,Logistic,dylan,sklearn.linear_model._logistic.LogisticRegression,mlflow_registry.ipynb,NOTEBOOK
1,27a2492b35504f2b935235960ba6bd42,9,FINISHED,/home/dylan/Bureau/ml-eng/mlflow_projets/mlrun...,2026-10-03 10:36:07.836000+00:00,2026-10-03 10:36:41.178000+00:00,0.500000,0.500000,0.263595,0.692787,...,NaN,NaN,None,prior,DummyClassifier,Dummyclassifier,dylan,sklearn.dummy.DummyClassifier,mlflow_registry.ipynb,NOTEBOOK


In [46]:
df.columns.tolist()

['run_id',
 'experiment_id',
 'status',
 'artifact_uri',
 'start_time',
 'end_time',
 'metrics.training_roc_auc',
 'metrics.roc_score',
 'metrics.training_precision_score',
 'metrics.training_log_loss',
 'metrics.training_recall_score',
 'metrics.training_score',
 'metrics.training_f1_score',
 'metrics.f1_score',
 'metrics.training_accuracy_score',
 'params.solver',
 'params.warm_start',
 'params.intercept_scaling',
 'params.tol',
 'params.max_iter',
 'params.penalty',
 'params.fit_intercept',
 'params.l1_ratio',
 'params.dual',
 'params.C',
 'params.n_jobs',
 'params.random_state',
 'params.verbose',
 'params.class_weight',
 'params.constant',
 'params.strategy',
 'tags.estimator_name',
 'tags.mlflow.runName',
 'tags.mlflow.user',
 'tags.estimator_class',
 'tags.mlflow.source.name',
 'tags.mlflow.source.type']

In [47]:
#  pour  avoir  tous  les runs avec les params et  metrics
runs =  mlflow.search_runs(
    experiment_ids=["9"] , 
   order_by=["metrics.f1_score DESC"] # tu  peux trier
)
runs[['run_id', 'metrics.f1_score' , 'tags.mlflow.runName' , 'metrics.roc_score' ]]

,run_id,metrics.f1_score,tags.mlflow.runName,metrics.roc_score
0,ec4eb4beb6ee48ab8855e585b0b02843,0.831169,Logistic,0.807143
1,27a2492b35504f2b935235960ba6bd42,0.677419,Dummyclassifier,0.500000


#   chargement  du  model  depuis  son  run_id  de mlflow

In [49]:
# Charger depuis MLflow Tracking via l'URI
# loaded_model = mlflow.sklearn.load_model(f"runs:/{run_id}/tracking_path")
try:
    new_model =  mlflow.sklearn.load_model(f"runs:/ec4eb4beb6ee48ab8855e585b0b02843/traking_path")
    print("model  bien  charge depuis  mlflow ui ")
    #   prediction  
    print("--------------quelque prediction------------- \n")
    print("valuer  predict  par  le model: " , new_model.predict(X_test[:4]))
    print("valuer  vrai :" , y_test[:4])

except:
    print("  model pas charge")

model  bien  charge depuis  mlflow ui 
--------------quelque prediction------------- 

valuer  predict  par  le model:  [0 0 0 1]
valuer  vrai : [0 1 0 1]


#   dernier  test    pour  voir  moi  meme 

In [60]:
mlflow.sklearn.autolog()
with  mlflow.start_run(run_name="Arbre de decision")  as run :
    tree =  DecisionTreeClassifier(random_state=42 ,max_depth=10)
    tree.fit(X_train  , y_train)
    y_pred = tree.predict(X_test)
    #   metrics 
    test_accuracy =  accuracy_score(y_test , y_pred)
    test_f1 =  f1_score(y_test , y_pred)
    test_roc_auc = roc_auc_score(y_test , y_pred)
    #   confusion  matrix
    plt.figure(figsize=(10,10))
    sns.heatmap(confusion_matrix(y_test ,  y_pred) , cmap="Blues" , fmt="d" , annot=True )
    plt.xlabel("predicted label")
    plt.ylabel("reel label")
    plt.title(f"confusion  matrix {tree}")
    plt.savefig("cf.png")
    plt.close()

    #  log  
    mlflow.log_metrics({
        "test_accuracy" : test_accuracy , 
        "test_f1" : test_f1 ,
        "test_roc_auc":test_roc_auc})
   # Modifiez la ligne 24 de votre code ainsi :
    mlflow.sklearn.log_model(
    sk_model=tree, 
    artifact_path="model_heath", 
    skops_trusted_types=["sklearn.tree._tree.Tree"]
)

    mlflow.log_artifact("cf.png")


    

    
    
    
    

2026/10/03 13:56:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/03 13:56:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/dylan/Bureau/ml-eng/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/10/03 13:57:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/03 13:57:20 WARNING mlflow.utils.environment: Failed to resolve installe

In [63]:
#  load  model 
try:
    new_tree =  mlflow.sklearn.load_model("runs:/e1820cc325b743cd9a095f3565ff301b/model_heath")
    print("model  bien  charge   depuis mlflow ui ..")
except:
    print("erreur  de chargement ..")

model  bien  charge   depuis mlflow ui ..


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples i

#   mlflow Models  Registry 

In [65]:
#   using  Mlflow   client  module  
from  mlflow import  MlflowClient

# create  an  instance

client = MlflowClient()

client

#   Registering  a model 

In [66]:
#  create  a Model  named  "Unicorn"
client.create_registered_model(name="unicorn")

<RegisteredModel: aliases={}, creation_timestamp=1791029926141, deployment_job_id=None, deployment_job_state=None, description=None, last_updated_timestamp=1791029926141, latest_versions=[], name='unicorn', tags={}, workspace='default'>

# searching  registry  models

client.search_registered_models(filter_string="MY_FILTER_STRING")
. identifiers   :  name   of the model   , tags -  tages  associated  with  model 
-  Comparators :  =   , != ,  LIKE , ILIKE 

# example search 

#   Experience 

In [67]:
mlflow.set_experiment("model_registry")
mlflow.set_experiment_tag("model" , "enregistre")

mlflow.sklearn.autolog()

with  mlflow.start_run(run_name="Dummyclassifier")  as run :
    model = DummyClassifier()
    model.fit(X_train , y_train)
    y_pred = model.predict(X_test)
    #   metrics
    f1 = f1_score(y_test , y_pred)
    roc_score =  roc_auc_score(y_test , y_pred)
    accuracy =  accuracy_score(y_test , y_pred)
    



2026/10/03 15:06:16 INFO mlflow.tracking.fluent: Experiment with name 'model_registry' does not exist. Creating a new experiment.
2026/10/03 15:06:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/03 15:06:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/dylan/Bureau/ml-eng/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

In [68]:
#   save  model  registry 
model_uri  =  "./mlruns/10/models/m-ee9d599811b1489e8088b2a69773515e/artifacts"
mlflow.register_model(model_uri=model_uri , name="unicorn")

Registered model 'unicorn' already exists. Creating a new version of this model...
Created version '1' of model 'unicorn'.


<ModelVersion: aliases=[], creation_timestamp=1791033522996, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1791033522996, metrics=None, model_id=None, name='unicorn', params=None, run_id=None, run_link=None, source='./mlruns/10/models/m-ee9d599811b1489e8088b2a69773515e/artifacts', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [72]:


mlflow.sklearn.autolog()

with  mlflow.start_run(run_name="Logistic")  as run :
    model = LogisticRegression()
    model.fit(X_train , y_train)
    y_pred = model.predict(X_test)
    #   metrics
    f1 = f1_score(y_test , y_pred)
    roc_score =  roc_auc_score(y_test , y_pred)
    accuracy =  accuracy_score(y_test , y_pred)

    mlflow.sklearn.log_model(model , "model_heath")




2026/10/03 15:25:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/03 15:25:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/dylan/Bureau/ml-eng/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/10/03 15:25:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/03 15:26:10 WARNING mlflow.utils.environment: Failed to resolve installe

In [75]:
run_id = mlflow.last_active_run().info.run_id
run_id

'04b63298dcba405dada045e0a29e8cd5'

In [77]:
#   registry  via  mlflow ui 
model_path =  f"runs:/{run_id}/model_heath"
mlflow.register_model(model_uri=model_path , name="unicorn")

Registered model 'unicorn' already exists. Creating a new version of this model...
2026/10/03 15:30:02 WARNING mlflow.tracking._model_registry.fluent: Run with id 04b63298dcba405dada045e0a29e8cd5 has no artifacts at artifact path 'model_heath', registering model based on models:/m-e62d58f8c28444a2ae03e2eed82e4db0 instead
Created version '2' of model 'unicorn'.


<ModelVersion: aliases=[], creation_timestamp=1791034202167, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1791034202167, metrics=None, model_id=None, name='unicorn', params=None, run_id='04b63298dcba405dada045e0a29e8cd5', run_link=None, source='models:/m-e62d58f8c28444a2ae03e2eed82e4db0', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [78]:
#  save  directement   dans registry
mlflow.sklearn.autolog()
with  mlflow.start_run(run_name="Arbre de decision")  as run :
    tree =  DecisionTreeClassifier()
    tree.fit(X_train  , y_train)
    y_pred = tree.predict(X_test)
    #   metrics 
    test_accuracy =  accuracy_score(y_test , y_pred)
    test_f1 =  f1_score(y_test , y_pred)
    test_roc_auc = roc_auc_score(y_test , y_pred)
    #   confusion  matrix
    plt.figure(figsize=(10,10))
    sns.heatmap(confusion_matrix(y_test ,  y_pred) , cmap="Blues" , fmt="d" , annot=True )
    plt.xlabel("predicted label")
    plt.ylabel("reel label")
    plt.title(f"confusion  matrix {tree}")
    plt.savefig("cf.png")
    plt.close()


    mlflow.sklearn.log_model(
    sk_model=tree, 
    artifact_path="model_heath", 
    skops_trusted_types=["sklearn.tree._tree.Tree"] , registered_model_name="unicorn"
)

    mlflow.log_artifact("cf.png")


2026/10/03 15:35:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/03 15:36:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/dylan/Bureau/ml-eng/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/10/03 15:36:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/03 15:36:42 WARNING mlflow.utils.environment: Failed to resolve installe

#   Models  stage 

In [81]:
from  mlflow import  MlflowClient
client =  MlflowClient()
#   trasition  vers  staging   (deprecier  )
client.transition_model_version_stage(name="unicorn" , stage="staging" ,version=3)

/tmp/ipykernel_11960/532384610.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(name="unicorn" , stage="staging" ,version=3)


<ModelVersion: aliases=[], creation_timestamp=1791034602583, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1791036240441, metrics=None, model_id=None, name='unicorn', params=None, run_id='bbc594cb93a04f72827815af4c27fe1e', run_link=None, source='models:/m-ffb45a9e55ac4bf4a9ab9fa6661fef2f', status='READY', status_message=None, tags={}, user_id=None, version=3, workspace='default'>

In [82]:
#   methode  recommande  
client.set_registered_model_alias(name="unicorn" , alias="challenger" , version=3)
print("ok ")

ok 


In [83]:
#   on  fait les tests
try:
    new_model =  mlflow.sklearn.load_model("models:/unicorn@challenger")
    print("bien  importer depuis models registry")

except:
    print("echec  de path !")

bien  importer depuis models registry


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of sampl

In [87]:
mlflow.sklearn.autolog()
with  mlflow.start_run(run_name="Randomforest")  as run :
    tree =  RandomForestClassifier()
    tree.fit(X_train  , y_train)
    y_pred = tree.predict(X_test)
    #   metrics 
    test_accuracy =  accuracy_score(y_test , y_pred)
    test_f1 =  f1_score(y_test , y_pred)
    test_roc_auc = roc_auc_score(y_test , y_pred)
    #   confusion  matrix
    plt.figure(figsize=(10,10))
    sns.heatmap(confusion_matrix(y_test ,  y_pred) , cmap="Blues" , fmt="d" , annot=True )
    plt.xlabel("predicted label")
    plt.ylabel("reel label")
    plt.title(f"confusion  matrix {tree}")
    plt.savefig("cf1.png")
    plt.close()


    mlflow.sklearn.log_model(
    sk_model=tree, 
    artifact_path="model_heath", 
    skops_trusted_types=["sklearn.tree._tree.Tree"] , registered_model_name="unicorn"
)

    mlflow.log_artifact("cf1.png")

2026/10/03 16:16:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/03 16:17:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/dylan/Bureau/ml-eng/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/10/03 16:17:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/03 16:17:49 WARNING mlflow.utils.environment: Failed to resolve installe